# sweep-hparam-distribution — ex2: build sweep distribution specs from (low, high, kind) tuples

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `sweep-hparam-distribution`. Running the final beacon cell reports progress against the `Config: sweep hparam distribution` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Config: sweep hparam distribution` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sweep-hparam-distribution`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sweep-hparam-distribution"
DD_SUBTOPIC = "Config: sweep hparam distribution"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Distribution registry — (low, high, kind) tuple to wandb spec

Ex1 hand-mapped hparam name → spec. The deepening move is a generic registry: given `(low, high, kind)`, emit the right distribution spec.

```python
spec = build_spec(low=1e-5, high=1e-1, kind='log_float')
# → {'distribution': 'log_uniform_values', 'min': 1e-5, 'max': 1e-1}
```

**Kinds:** `'float'` → `'uniform'`; `'log_float'` → `'log_uniform_values'`; `'int'` → `'int_uniform'`. Unknown kind raises `ValueError`.

**Why a registry over named-hparam dispatch.** When you sweep 30 hparams across 3 projects, hand-listing each name is fragile. A `(low, high, kind)` tuple keeps the SHAPE separate from the NAME — projects share kind logic.

**Validation: low < high.** A flipped range is the most common typo. Detect it up front; downstream wandb won't give a helpful error.

### Exercise 2 — build sweep distribution specs from (low, high, kind) tuples

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply a kind-dispatch table (`'float' | 'log_float' | 'int'`) to convert `(low, high, kind)` tuples into the correct wandb distribution spec dicts, validating that `low < high` and raising `ValueError` on unknown kinds.
> Keywords: distribution, registry, tuple-dispatch, validation
> ```

**KCs targeted:** `kind-to-distribution-dispatch`, `range-validation-low-less-than-high`

Implement `ex2_build_distribution_spec(low, high, kind)`. A generic registry over the ex1 mapping.

Kind → distribution:
- `'float'` → `{'distribution': 'uniform', 'min': low, 'max': high}`
- `'log_float'` → `{'distribution': 'log_uniform_values', 'min': low, 'max': high}`
- `'int'` → `{'distribution': 'int_uniform', 'min': low, 'max': high}`

Validation:
1. `low < high` — strict. If violated: raise `ValueError` whose message contains both `'low'` and `'high'` (case-insensitive).
2. `kind in {'float', 'log_float', 'int'}` — else raise `ValueError` whose message contains the offending kind.
3. For `'log_float'` only: BOTH `low > 0` and `high > 0`. Otherwise log-uniform is undefined — raise `ValueError` containing `'log'`.
4. For `'int'`: `low` and `high` must both be `int`. If either is a non-int (including bool), raise `TypeError` containing `'int'`.

Output: `dict` matching wandb's distribution spec schema.

In [ ]:
def ex2_build_distribution_spec(low, high, kind):
    if kind not in {'float', 'log_float', 'int'}:
        raise ValueError(
            f'unknown kind {kind!r}, must be one of float/log_float/int'
        )
    if kind == 'int':
        # bool is a subclass of int; exclude it explicitly.
        if not (type(low) is int and type(high) is int):
            raise TypeError(
                f'kind=int requires int bounds, got low={type(low).__name__} high={type(high).__name__}'
            )
    if not (low < high):
        raise ValueError(f'low must be < high, got low={low} high={high}')
    if kind == 'log_float' and (low <= 0 or high <= 0):
        raise ValueError(
            f'log distribution requires low>0 and high>0, got low={low} high={high}'
        )
    dist_name = {
        'float': 'uniform',
        'log_float': 'log_uniform_values',
        'int': 'int_uniform',
    }[kind]
    return {'distribution': dist_name, 'min': low, 'max': high}


<details><summary>Solution</summary>

```python
def ex2_build_distribution_spec(low, high, kind):
    if kind not in {'float', 'log_float', 'int'}:
        raise ValueError(
            f'unknown kind {kind!r}, must be one of float/log_float/int'
        )
    if kind == 'int':
        # bool is a subclass of int; exclude it explicitly.
        if not (type(low) is int and type(high) is int):
            raise TypeError(
                f'kind=int requires int bounds, got low={type(low).__name__} high={type(high).__name__}'
            )
    if not (low < high):
        raise ValueError(f'low must be < high, got low={low} high={high}')
    if kind == 'log_float' and (low <= 0 or high <= 0):
        raise ValueError(
            f'log distribution requires low>0 and high>0, got low={low} high={high}'
        )
    dist_name = {
        'float': 'uniform',
        'log_float': 'log_uniform_values',
        'int': 'int_uniform',
    }[kind]
    return {'distribution': dist_name, 'min': low, 'max': high}
```

**Validation order matters.** Check kind FIRST — otherwise an unknown kind with bad bounds raises the bound error instead of the more informative kind error.

**`type(low) is int` not `isinstance(low, int)`.** `bool` is a subclass of `int` in Python — `isinstance(True, int) == True`. The stricter `type() is int` check rejects booleans, which is almost always what you want for a bounds value.

**Why a fresh dict per call.** Returning a shared dict (e.g. from a cached table) breaks downstream code that mutates the spec — the next call gets the mutated version. Constructing a new dict literal sidesteps the issue without needing `copy.deepcopy`.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()